In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
from torch.utils.data import TensorDataset, Subset

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

In [32]:
class HadamardTransform:

    def __init__(self, target_len=784):
        self.target_len = target_len
        n_qubits = int(np.ceil(np.log2(target_len)))
        self.N = 2**n_qubits

        qc = QuantumCircuit(n_qubits)
        qc.h(range(n_qubits))

        self.H = np.asarray(Operator(qc).data, dtype=np.complex64)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if x.is_cuda:
            x = x.cpu()

        img_flat = x.flatten().detach().numpy().astype(np.float32)
        L = img_flat.size

        state = np.zeros(self.N, dtype=np.complex64)
        state[:L] = img_flat.astype(np.complex64)

        norm = np.linalg.norm(state)
        if norm == 0:
            return torch.zeros_like(x)

        state /= norm

        out = self.H @ state
        y = np.real(out[:L]) * norm
        y = y.reshape(x.shape).astype(np.float32)

        return torch.from_numpy(y)

In [33]:
# download format
# turns MNIST images to PyTorch tensors and normalizes between [-1,1] centered at 0
transform = transforms.Compose([
    transforms.RandomCrop(28, padding=3),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    HadamardTransform()
])

# download data
raw_train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
raw_test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

In [34]:
# apply ht to all images before training so that i dont run qiskit every epoch
ht = HadamardTransform() # ht qiskit class

def precompute(dataset):

    temp_image_list, temp_label_list = [], []

    for image, label in DataLoader(dataset, batch_size=1, shuffle=False):
        image_ht = ht(image[0])                         # get rid of batch dimensions so i can apply the ht
        temp_image_list.append(image_ht.unsqueeze(0))   # undo previous step and append to temp_image_list
        temp_label_list.append(label)

    concatinated_image_dataset = torch.cat(temp_image_list, dim=0) # combines all tensors in temp lists to a single tensor with batch dimensions
    concatinated_label_dataset = torch.cat(temp_label_list, dim=0)
    return TensorDataset(concatinated_image_dataset, concatinated_label_dataset)

# sanity test lines to make sure the cnn works
# it works but it takes a super long time to run all images in the fashion mnist
raw_train_subset = Subset(raw_train_dataset, range(30000))
raw_test_subset = Subset(raw_test_dataset, range(5000))

# use raw_train_small and raw_test_small to decrease number of training and test images
# use raw_train_dataset and raw_test_dataset for entire dataset
train_dataset = precompute(raw_train_subset)
test_dataset  = precompute(raw_test_subset)

In [35]:
# loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,         # each epoch is 128 samples
    shuffle=True            # randomize after each training epoch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,         # each epoch is 256 samples
    shuffle=False
)

# test shapes of pytorch datasets
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([128, 1, 28, 28])
torch.Size([128])


In [36]:
# CNN model
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)

        self.drop = nn.Dropout(0.25)

        self.fc1 = nn.Linear(128 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.drop(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# create model, loss, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()

muon_params = []
adamw_params = []

for name, p in model.named_parameters():
    
    if not p.requires_grad:
        continue

    if p.ndim == 2 and name.endswith("weight") and name != "fc2.weight":
        muon_params.append(p)
    else:
        adamw_params.append(p)

optimizer_muon = torch.optim.Muon(muon_params, lr=0.001, weight_decay=0.0)
optimizer_adamw = torch.optim.AdamW(adamw_params, lr=0.001, weight_decay=0.01) # for non-2D params

In [37]:
# training loop
num_epochs = 25
print("Muon", len(muon_params))
print("Adam", len(adamw_params))

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_muon.zero_grad()
        optimizer_adamw.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer_muon.step()
        optimizer_adamw.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

Muon 1
Adam 15
Epoch [1/25], Loss: 0.8459
Epoch [2/25], Loss: 0.4930
Epoch [3/25], Loss: 0.4079
Epoch [4/25], Loss: 0.3548
Epoch [5/25], Loss: 0.3095
Epoch [6/25], Loss: 0.2702
Epoch [7/25], Loss: 0.2379
Epoch [8/25], Loss: 0.2053
Epoch [9/25], Loss: 0.1777
Epoch [10/25], Loss: 0.1515
Epoch [11/25], Loss: 0.1307
Epoch [12/25], Loss: 0.1039
Epoch [13/25], Loss: 0.0868
Epoch [14/25], Loss: 0.0715
Epoch [15/25], Loss: 0.0609
Epoch [16/25], Loss: 0.0466
Epoch [17/25], Loss: 0.0432
Epoch [18/25], Loss: 0.0370
Epoch [19/25], Loss: 0.0336
Epoch [20/25], Loss: 0.0275
Epoch [21/25], Loss: 0.0235
Epoch [22/25], Loss: 0.0241
Epoch [23/25], Loss: 0.0240
Epoch [24/25], Loss: 0.0197
Epoch [25/25], Loss: 0.0208


In [ ]:
# results and accuracy
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 85.96%


: 